# Lesson 12D: GANs - Practical

Train GAN on 2D moon dataset.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons

torch.manual_seed(42)
print('✅ Loaded!')

In [ ]:
class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 2))
    def forward(self, z):
        return self.net(z)

class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(2, 32), nn.LeakyReLU(0.2), nn.Linear(32, 1), nn.Sigmoid())
    def forward(self, x):
        return self.net(x)

G = Generator()
D = Discriminator()
criterion = nn.BCELoss()
opt_G = optim.Adam(G.parameters(), lr=0.001)
opt_D = optim.Adam(D.parameters(), lr=0.001)
print('✅ GAN initialized')

In [ ]:
# Train
X_real, _ = make_moons(n_samples=1000, noise=0.1, random_state=42)
X_real = torch.FloatTensor(X_real)

for epoch in range(500):
    # Train D
    z = torch.randn(64, 10)
    fake = G(z)
    D_real = D(X_real[:64])
    D_fake = D(fake.detach())
    loss_D = criterion(D_real, torch.ones(64, 1)) + criterion(D_fake, torch.zeros(64, 1))
    opt_D.zero_grad()
    loss_D.backward()
    opt_D.step()
    
    # Train G
    z = torch.randn(64, 10)
    fake = G(z)
    loss_G = criterion(D(fake), torch.ones(64, 1))
    opt_G.zero_grad()
    loss_G.backward()
    opt_G.step()
    
    if (epoch+1) % 100 == 0:
        print(f'Epoch {epoch+1}, D Loss: {loss_D.item():.3f}, G Loss: {loss_G.item():.3f}')

print('✅ GAN trained!')